# Exploratory Data Analysis

SQL-first validation of MetaPro RPKM sample data against taxonomy/pathway reference tables.

- [Data model & logical ER diagram](../docs/data-model.md)
- [Setup & workflow](../README.md)

In [1]:
from pathlib import Path

import duckdb

%load_ext sql

REPO_ROOT = Path("..").resolve().parents[1]  # notebooks → exploration → analytics → repo root
PARQUET_DIR = REPO_ROOT / "resources/db/parquet"
RPKM_FILES = [
    REPO_ROOT / "resources/example_data/test_rpkm_1.tsv",
    REPO_ROOT / "resources/example_data/test_rpkm_2.tsv",
]
TABLES = [
    "names", "nodes", "parents",
    "pathway_nodes", "pathway_edges",
    "pathway_superpathways", "superpathways",
]
KEY_COLS = ["GeneID", "Length", "Reads", "EC#", "RPKM", "Unclassified"]

conn = duckdb.connect()
%sql conn --alias duckdb

print(f"Repo root: {REPO_ROOT}")

The 'toml' package isn't installed. To load settings from pyproject.toml or ~/.jupysql/config, install with: pip install toml

Repo root: /Users/sibyl/study/metapro-data-vis/.worktrees/exploration-eda


In [2]:
import sys

missing = []
for table in TABLES:
    p = PARQUET_DIR / f"{table}.parquet"
    if not p.exists():
        missing.append(str(p))
for rpkm in RPKM_FILES:
    if not rpkm.exists():
        missing.append(str(rpkm))

if missing:
    print("MISSING INPUT FILES:")
    for m in missing:
        print(f"  - {m}")
    print("\nRun: uv run python exploration/scripts/export_parquet.py")
    sys.exit(1)

for table in TABLES:
    p = PARQUET_DIR / f"{table}.parquet"
    n = conn.execute(f"SELECT COUNT(*) FROM '{p}'").fetchone()[0]
    mb = p.stat().st_size / (1024 * 1024)
    print(f"{table}: {n:,} rows ({mb:.1f} MB)")

for rpkm in RPKM_FILES:
    mb = rpkm.stat().st_size / (1024 * 1024)
    print(f"{rpkm.name}: {mb:.1f} MB")

print("All inputs present.")

names: 2,840,139 rows (137.3 MB)
nodes: 2,840,139 rows (10.8 MB)
parents: 2,840,134 rows (129.7 MB)
pathway_nodes: 23,864 rows (1.0 MB)
pathway_edges: 46,726 rows (2.5 MB)
pathway_superpathways: 193 rows (0.0 MB)
superpathways: 13 rows (0.0 MB)
test_rpkm_1.tsv: 51.0 MB
test_rpkm_2.tsv: 71.1 MB
All inputs present.


## 3. Data Dictionary

Register the exported reference Parquet files as DuckDB views, then inspect schemas, row counts, and the superpathways name list (`id` is a build-time `uuid4()` surrogate — see `make_superpathway_db.ipynb`).


In [3]:
for table in TABLES:
    conn.execute(f"""
        CREATE OR REPLACE VIEW {table} AS
        SELECT * FROM '{PARQUET_DIR / f"{table}.parquet"}'
    """)

print("Views created for all reference tables.")

Views created for all reference tables.


In [4]:
# Data dictionary is ~28 rows; JupySQL default displaylimit is 10.
%config SqlMagic.displaylimit = 0

In [5]:
%%sql
SELECT 'names' AS tbl, column_name, column_type FROM (DESCRIBE names)
UNION ALL SELECT 'nodes', column_name, column_type FROM (DESCRIBE nodes)
UNION ALL SELECT 'parents', column_name, column_type FROM (DESCRIBE parents)
UNION ALL SELECT 'pathway_nodes', column_name, column_type FROM (DESCRIBE pathway_nodes)
UNION ALL SELECT 'pathway_edges', column_name, column_type FROM (DESCRIBE pathway_edges)
UNION ALL SELECT 'pathway_superpathways', column_name, column_type FROM (DESCRIBE pathway_superpathways)
UNION ALL SELECT 'superpathways', column_name, column_type FROM (DESCRIBE superpathways)
ORDER BY tbl, column_name;

Running query in 'duckdb'

tbl,column_name,column_type
names,id,VARCHAR
names,name,VARCHAR
names,tax_id,BIGINT
nodes,id,BIGINT
parents,id,VARCHAR
parents,t_class,BIGINT
parents,t_family,BIGINT
parents,t_genus,BIGINT
parents,t_kingdom,BIGINT
parents,t_order,BIGINT


In [6]:
%config SqlMagic.displaylimit = 10

In [7]:
%%sql
SELECT 'names' AS tbl, COUNT(*) AS n FROM names
UNION ALL SELECT 'nodes', COUNT(*) FROM nodes
UNION ALL SELECT 'parents', COUNT(*) FROM parents
UNION ALL SELECT 'pathway_nodes', COUNT(*) FROM pathway_nodes
UNION ALL SELECT 'pathway_edges', COUNT(*) FROM pathway_edges
UNION ALL SELECT 'pathway_superpathways', COUNT(*) FROM pathway_superpathways
UNION ALL SELECT 'superpathways', COUNT(*) FROM superpathways
ORDER BY tbl;

Running query in 'duckdb'

tbl,n
names,2840139
nodes,2840139
parents,2840134
pathway_edges,46726
pathway_nodes,23864
pathway_superpathways,193
superpathways,13


In [8]:
# superpathways has 13 rows; default displaylimit is 10.
%config SqlMagic.displaylimit = 0


In [9]:
%%sql
-- superpathways names (§3); id is str(uuid4()) from make_superpathway_db.ipynb at build time
SELECT name
FROM superpathways
ORDER BY name;


Running query in 'duckdb'

name
Amino acid metabolism
Biosynthesis of other secondary metabolites
Carbohydrate metabolism
Chemical structure transformation maps
Energy metabolism
Global and overview maps
Glycan biosynthesis and metabolism
Lipid metabolism
Metabolism of cofactors and vitamins
Metabolism of other amino acids


In [10]:
%config SqlMagic.displaylimit = 10


## 4. Cardinality

Measure reference-table cardinalities, graph degree distributions, and internal pathway join rates that downstream joins rely on.


In [11]:
%%sql
SELECT
    MIN(name_count) AS min_names,
    MAX(name_count) AS max_names,
    MEDIAN(name_count) AS median_names,
    AVG(name_count) AS avg_names
FROM (
    SELECT tax_id, COUNT(*) AS name_count
    FROM names
    GROUP BY tax_id
);

Running query in 'duckdb'

min_names,max_names,median_names,avg_names
1,1,1.0,1.0


In [12]:
%%sql
WITH out_degree AS (
    SELECT source AS node_id, COUNT(*) AS out_deg
    FROM pathway_edges
    GROUP BY source
),
in_degree AS (
    SELECT target AS node_id, COUNT(*) AS in_deg
    FROM pathway_edges
    GROUP BY target
),
all_nodes AS (
    SELECT id AS node_id FROM pathway_nodes
)
SELECT
    'out_degree' AS metric,
    MIN(COALESCE(o.out_deg, 0)) AS min,
    MAX(COALESCE(o.out_deg, 0)) AS max,
    MEDIAN(COALESCE(o.out_deg, 0)) AS median
FROM all_nodes n LEFT JOIN out_degree o ON n.node_id = o.node_id
UNION ALL
SELECT
    'in_degree',
    MIN(COALESCE(i.in_deg, 0)),
    MAX(COALESCE(i.in_deg, 0)),
    MEDIAN(COALESCE(i.in_deg, 0))
FROM all_nodes n LEFT JOIN in_degree i ON n.node_id = i.node_id;

Running query in 'duckdb'

metric,min,max,median
out_degree,0,945,0.0
in_degree,0,945,0.0


In [13]:
%%sql
SELECT
    MIN(edge_count) AS min_edges,
    MAX(edge_count) AS max_edges,
    MEDIAN(edge_count) AS median_edges
FROM (
    SELECT pathway, COUNT(*) AS edge_count
    FROM pathway_edges
    GROUP BY pathway
);

Running query in 'duckdb'

min_edges,max_edges,median_edges
2,2410,132.0


In [14]:
%%sql
-- Reverse cardinalities for ER edge evidence (§4)
SELECT 'psp_per_superpathway' AS metric,
       MIN(n) AS min_val, MAX(n) AS max_val, MEDIAN(n) AS median_val
FROM (SELECT superpathway, COUNT(*) AS n FROM pathway_superpathways GROUP BY superpathway)
UNION ALL
SELECT 'parents_per_kingdom_node', MIN(n), MAX(n), MEDIAN(n)
FROM (SELECT t_kingdom, COUNT(*) AS n FROM parents WHERE t_kingdom IS NOT NULL GROUP BY t_kingdom)
UNION ALL
SELECT 'superpathways_without_psp', COUNT(*), NULL, NULL
FROM superpathways sp
LEFT JOIN pathway_superpathways psp ON psp.superpathway = sp.id
WHERE psp.id IS NULL;


Running query in 'duckdb'

metric,min_val,max_val,median_val
psp_per_superpathway,2,31,14.0
parents_per_kingdom_node,1,1322687,1204.0
superpathways_without_psp,0,None,None


In [15]:
%%sql
-- pathway_nodes.pathway -> pathway_superpathways (+ superpathways) orphan counts (§4)
SELECT
  (SELECT COUNT(*) FROM pathway_nodes pn
   LEFT JOIN pathway_superpathways psp ON pn.pathway = psp.id
   WHERE psp.id IS NULL) AS orphan_pathway_to_psp,
  (SELECT COUNT(*) FROM pathway_nodes pn
   LEFT JOIN pathway_superpathways psp ON pn.pathway = psp.id
   LEFT JOIN superpathways sp ON psp.superpathway = sp.id
   WHERE psp.id IS NULL OR sp.id IS NULL) AS orphan_full_chain;
-- expect 0, 0



Running query in 'duckdb'

orphan_pathway_to_psp,orphan_full_chain
0,0


In [16]:
%%sql
-- pathway_nodes.pathway -> pathway_superpathways cardinality (§4)
WITH per_pathway AS (
  SELECT pathway, COUNT(*) AS node_count
  FROM pathway_nodes
  GROUP BY pathway
)
SELECT
  (SELECT COUNT(*) FROM pathway_nodes) AS pathway_nodes_total,
  (SELECT COUNT(DISTINCT pathway) FROM pathway_nodes) AS distinct_pathway_ids,
  (SELECT COUNT(*) FROM pathway_superpathways) AS pathway_superpathways_rows,
  (SELECT COUNT(*) FROM pathway_superpathways psp
   LEFT JOIN pathway_nodes pn ON pn.pathway = psp.id
   WHERE pn.id IS NULL) AS psp_without_nodes,
  (SELECT MIN(node_count) FROM per_pathway) AS min_nodes_per_pathway,
  (SELECT MAX(node_count) FROM per_pathway) AS max_nodes_per_pathway,
  (SELECT MEDIAN(node_count) FROM per_pathway) AS median_nodes_per_pathway;



Running query in 'duckdb'

pathway_nodes_total,distinct_pathway_ids,pathway_superpathways_rows,psp_without_nodes,min_nodes_per_pathway,max_nodes_per_pathway,median_nodes_per_pathway
23864,170,193,23,1,3716,91.5


## 5. RPKM Profiling

Load both sample RPKM TSV files as DuckDB views, normalize EC values, and profile tax_id column coverage and sparsity.

In [17]:
for i, rpkm_path in enumerate(RPKM_FILES, start=1):
    conn.execute(f"""
        CREATE OR REPLACE VIEW rpkm_{i} AS
        SELECT * FROM read_csv('{rpkm_path}', delim='\t', header=true, auto_detect=true)
    """)
    n = conn.execute(f"SELECT COUNT(*) FROM rpkm_{i}").fetchone()[0]
    print(f"rpkm_{i} ({rpkm_path.name}): {n:,} rows")

rpkm_1 (test_rpkm_1.tsv): 425,828 rows
rpkm_2 (test_rpkm_2.tsv): 461,112 rows


In [18]:
%%sql
SELECT
    "EC#" AS ec_raw,
    CASE
        WHEN "EC#" IS NULL OR TRIM(CAST("EC#" AS VARCHAR)) IN ('', 'None', 'none', 'NA', 'null') THEN '0.0.0.0'
        WHEN STARTS_WITH(CAST("EC#" AS VARCHAR), 'EC:') THEN SUBSTR(CAST("EC#" AS VARCHAR), 4)
        ELSE CAST("EC#" AS VARCHAR)
    END AS ec_normalized,
    COUNT(*) AS row_count
FROM rpkm_1
GROUP BY 1, 2
ORDER BY row_count DESC
LIMIT 20;

Running query in 'duckdb'

ec_raw,ec_normalized,row_count
None,0.0.0.0,235991
EC:2.7.13.3,2.7.13.3,8513
EC:3.6.4.12,3.6.4.12,4279
EC:2.7.7.6,2.7.7.6,3466
EC:2.3.2.27,2.3.2.27,2987
EC:2.7.11.1,2.7.11.1,2672
EC:2.7.7.7,2.7.7.7,2658
EC:3.6.3.14,3.6.3.14,2645
EC:2.7.1.69,2.7.1.69,2412
EC:5.2.1.8,5.2.1.8,2128


In [19]:
cols = conn.execute("SELECT * FROM rpkm_1 LIMIT 0").description
all_cols = [c[0] for c in cols]
tax_cols = [c for c in all_cols if c not in KEY_COLS]
print(f"Tax_id columns: {len(tax_cols)}")

# DuckDB UNPIVOT needs the dynamic tax_id column list generated in Python.
tax_cols_sql = ", ".join(f'"{c}"' for c in tax_cols)
conn.execute(f"""
    CREATE OR REPLACE VIEW rpkm_1_long AS
    UNPIVOT rpkm_1
    ON {tax_cols_sql}
    INTO NAME tax_id VALUE rpkm_value
""")
print("rpkm_1 sparsity:", conn.execute("""
    SELECT
        COUNT(*) AS total_cells,
        COUNT(CASE WHEN rpkm_value > 0 THEN 1 END) AS nonzero_cells,
        ROUND(100.0 * COUNT(CASE WHEN rpkm_value > 0 THEN 1 END) / COUNT(*), 2) AS pct_nonzero
    FROM rpkm_1_long
""").fetchone())

Tax_id columns: 8


rpkm_1 sparsity: (3406624, 59439, 1.74)


In [20]:
tax_set_1 = set(tax_cols)
cols2 = [c[0] for c in conn.execute("SELECT * FROM rpkm_2 LIMIT 0").description]
tax_set_2 = set(c for c in cols2 if c not in KEY_COLS)
print(f"rpkm_1 tax columns: {len(tax_set_1)}")
print(f"rpkm_2 tax columns: {len(tax_set_2)}")
print(f"intersection: {len(tax_set_1 & tax_set_2)}")
print(f"only in rpkm_1: {len(tax_set_1 - tax_set_2)}")
print(f"only in rpkm_2: {len(tax_set_2 - tax_set_1)}")

rpkm_1 tax columns: 8
rpkm_2 tax columns: 12
intersection: 6
only in rpkm_1: 2
only in rpkm_2: 6


## 6. Reference Integrity

Check internal reference-table relationships: orphan FKs (including all `parents` rank columns), parentless-node allowlist, rank ladder completeness, rank transitive consistency, rank placement categorization (finest filled rank; at-rank vs sub-rank), duplicate pathway EC names, `pathway_nodes` graphics-type and EC-dotted name overlap, and pathway nodes without graph edges.


In [21]:
# Orphan-FK UNION returns 12 rows; default displaylimit is 10.
%config SqlMagic.displaylimit = 0

In [22]:
%%sql
SELECT 'names.tax_id -> nodes' AS fk,
       COUNT(*) AS orphan_count
FROM names n
LEFT JOIN nodes nd ON n.tax_id = nd.id
WHERE nd.id IS NULL
UNION ALL
SELECT 'parents.tax_id -> nodes', COUNT(*)
FROM parents p LEFT JOIN nodes nd ON p.tax_id = nd.id
WHERE nd.id IS NULL
UNION ALL
SELECT 'parents.t_kingdom -> nodes', COUNT(*)
FROM parents p LEFT JOIN nodes nd ON p.t_kingdom = nd.id
WHERE p.t_kingdom IS NOT NULL AND nd.id IS NULL
UNION ALL
SELECT 'parents.t_phylum -> nodes', COUNT(*)
FROM parents p LEFT JOIN nodes nd ON p.t_phylum = nd.id
WHERE p.t_phylum IS NOT NULL AND nd.id IS NULL
UNION ALL
SELECT 'parents.t_class -> nodes', COUNT(*)
FROM parents p LEFT JOIN nodes nd ON p.t_class = nd.id
WHERE p.t_class IS NOT NULL AND nd.id IS NULL
UNION ALL
SELECT 'parents.t_order -> nodes', COUNT(*)
FROM parents p LEFT JOIN nodes nd ON p.t_order = nd.id
WHERE p.t_order IS NOT NULL AND nd.id IS NULL
UNION ALL
SELECT 'parents.t_family -> nodes', COUNT(*)
FROM parents p LEFT JOIN nodes nd ON p.t_family = nd.id
WHERE p.t_family IS NOT NULL AND nd.id IS NULL
UNION ALL
SELECT 'parents.t_genus -> nodes', COUNT(*)
FROM parents p LEFT JOIN nodes nd ON p.t_genus = nd.id
WHERE p.t_genus IS NOT NULL AND nd.id IS NULL
UNION ALL
SELECT 'parents.t_species -> nodes', COUNT(*)
FROM parents p LEFT JOIN nodes nd ON p.t_species = nd.id
WHERE p.t_species IS NOT NULL AND nd.id IS NULL
UNION ALL
SELECT 'pathway_edges.source -> pathway_nodes', COUNT(*)
FROM pathway_edges e LEFT JOIN pathway_nodes pn ON e.source = pn.id
WHERE pn.id IS NULL
UNION ALL
SELECT 'pathway_edges.target -> pathway_nodes', COUNT(*)
FROM pathway_edges e LEFT JOIN pathway_nodes pn ON e.target = pn.id
WHERE pn.id IS NULL
UNION ALL
SELECT 'pathway_superpathways.superpathway -> superpathways', COUNT(*)
FROM pathway_superpathways ps LEFT JOIN superpathways sp ON ps.superpathway = sp.id
WHERE sp.id IS NULL;


Running query in 'duckdb'

fk,orphan_count
names.tax_id -> nodes,0
parents.tax_id -> nodes,0
parents.t_kingdom -> nodes,0
parents.t_phylum -> nodes,0
parents.t_class -> nodes,0
parents.t_order -> nodes,0
parents.t_family -> nodes,0
parents.t_genus -> nodes,0
parents.t_species -> nodes,0
pathway_edges.source -> pathway_nodes,0


In [23]:
%config SqlMagic.displaylimit = 10

In [24]:
%%sql
-- nodes without parents: exactly 5 known meta/root taxons (§6 allowlist)
SELECT n.id AS tax_id, nm.name AS scientific_name
FROM nodes n
LEFT JOIN parents p ON n.id = p.tax_id
LEFT JOIN names nm ON nm.tax_id = n.id
WHERE p.tax_id IS NULL
ORDER BY n.id;

Running query in 'duckdb'

tax_id,scientific_name
1,root
10239,Viruses
131567,cellular organisms
2787823,unclassified entries
2787854,other entries


In [25]:
%%sql
-- parentless allowlist regression (expect parentless_count=5, unexpected_parentless=0, missing_allowed=0)
SELECT
  (SELECT COUNT(*) FROM nodes n
   LEFT JOIN parents p ON n.id = p.tax_id WHERE p.tax_id IS NULL) AS parentless_count,
  (SELECT COUNT(*) FROM nodes n
   LEFT JOIN parents p ON n.id = p.tax_id
   WHERE p.tax_id IS NULL
     AND n.id NOT IN (1, 10239, 131567, 2787823, 2787854)) AS unexpected_parentless,
  (SELECT COUNT(*) FROM (
     SELECT unnest([1::BIGINT, 10239, 131567, 2787823, 2787854]) AS tax_id
   ) allowed
   WHERE tax_id NOT IN (
     SELECT n.id FROM nodes n
     LEFT JOIN parents p ON n.id = p.tax_id WHERE p.tax_id IS NULL
   )) AS missing_allowed;

Running query in 'duckdb'

parentless_count,unexpected_parentless,missing_allowed
5,0,0


In [26]:
%%sql
-- parents duplicate tax_id (expect 0; schema declares UNIQUE)
SELECT tax_id, COUNT(*) AS n
FROM parents
GROUP BY tax_id
HAVING COUNT(*) > 1;

Running query in 'duckdb'

tax_id,n


In [27]:
%%sql
SELECT name, COUNT(*) AS n
FROM pathway_nodes
GROUP BY name
HAVING COUNT(*) > 1
ORDER BY n DESC
LIMIT 20;

Running query in 'duckdb'

name,n
1.14.14.1,77
Glycolysis / Gluconeogenesis,49
C00024,48
C00022,48
Citrate cycle (TCA cycle),46
Pyruvate metabolism,35
C00083,35
4.2.1.17,33
2.3.1.85,33
"Glycine, serine and threonine metabolism",32


In [28]:
%%sql
-- pathway_nodes.type is KGML graphics shape, not semantic enzyme/compound role (§6)
SELECT type, COUNT(*) AS n
FROM pathway_nodes
GROUP BY 1
ORDER BY n DESC;


Running query in 'duckdb'

type,n
circle,15403
rectangle,7169
roundrectangle,1292


In [29]:
%%sql
-- EC-dotted names (1-4 numeric segments) vs rectangle type; RPKM fallback token check (§6)
WITH ec AS (
  SELECT * FROM pathway_nodes
  WHERE regexp_matches(name, '^[0-9]+(\.[0-9]+){0,3}$')
),
rect AS (
  SELECT * FROM pathway_nodes WHERE type = 'rectangle'
)
SELECT
  (SELECT COUNT(*) FROM ec) AS ec_dotted_rows,
  (SELECT COUNT(DISTINCT name) FROM ec) AS ec_dotted_distinct_names,
  (SELECT COUNT(*) FROM rect) AS rectangle_rows,
  (SELECT COUNT(DISTINCT name) FROM rect) AS rectangle_distinct_names,
  (SELECT COUNT(*) FROM ec WHERE type != 'rectangle') AS ec_not_rectangle_rows,
  (SELECT COUNT(*) FROM rect
   WHERE NOT regexp_matches(name, '^[0-9]+(\.[0-9]+){0,3}$')) AS rectangle_not_ec_rows,
  (SELECT COUNT(*) FROM pathway_nodes WHERE name = '0.0.0.0') AS fallback_token_in_reference;



Running query in 'duckdb'

ec_dotted_rows,ec_dotted_distinct_names,rectangle_rows,rectangle_distinct_names,ec_not_rectangle_rows,rectangle_not_ec_rows,fallback_token_in_reference
7064,3867,7169,3942,0,105,0


In [30]:
%%sql
-- EC-dotted segment depth (§6); non-EC rectangle names are compound ids or ellipsis labels
SELECT
  CASE
    WHEN regexp_matches(name, '^[0-9]+$') THEN '1'
    WHEN regexp_matches(name, '^[0-9]+\.[0-9]+$') THEN '2'
    WHEN regexp_matches(name, '^[0-9]+\.[0-9]+\.[0-9]+$') THEN '3'
    WHEN regexp_matches(name, '^[0-9]+\.[0-9]+\.[0-9]+\.[0-9]+$') THEN '4'
  END AS ec_segments,
  COUNT(*) AS ec_dotted_rows
FROM pathway_nodes
WHERE regexp_matches(name, '^[0-9]+(\.[0-9]+){0,3}$')
GROUP BY 1
ORDER BY 1;



Running query in 'duckdb'

ec_segments,ec_dotted_rows
4,7064


In [31]:
%%sql
SELECT
    COUNT(*) AS total,
    COUNT(CASE WHEN t_species IS NOT NULL AND (
        t_genus IS NULL OR t_family IS NULL OR t_order IS NULL
        OR t_class IS NULL OR t_phylum IS NULL OR t_kingdom IS NULL
    ) THEN 1 END) AS species_missing_upstream,
    COUNT(CASE WHEN t_genus IS NOT NULL AND (
        t_family IS NULL OR t_order IS NULL OR t_class IS NULL
        OR t_phylum IS NULL OR t_kingdom IS NULL
    ) THEN 1 END) AS genus_missing_upstream,
    COUNT(CASE WHEN t_family IS NOT NULL AND (
        t_order IS NULL OR t_class IS NULL
        OR t_phylum IS NULL OR t_kingdom IS NULL
    ) THEN 1 END) AS family_missing_upstream,
    COUNT(CASE WHEN t_order IS NOT NULL AND (
        t_class IS NULL OR t_phylum IS NULL OR t_kingdom IS NULL
    ) THEN 1 END) AS order_missing_upstream,
    COUNT(CASE WHEN t_class IS NOT NULL AND (
        t_phylum IS NULL OR t_kingdom IS NULL
    ) THEN 1 END) AS class_missing_upstream,
    COUNT(CASE WHEN t_phylum IS NOT NULL AND t_kingdom IS NULL THEN 1 END) AS phylum_missing_upstream,
    COUNT(CASE WHEN t_kingdom IS NOT NULL AND (
        t_phylum IS NOT NULL OR t_class IS NOT NULL OR t_order IS NOT NULL
        OR t_family IS NOT NULL OR t_genus IS NOT NULL OR t_species IS NOT NULL
    ) AND (t_phylum IS NULL) THEN 1 END) AS kingdom_with_finer_but_missing_phylum
FROM parents;

Running query in 'duckdb'

total,species_missing_upstream,genus_missing_upstream,family_missing_upstream,order_missing_upstream,class_missing_upstream,phylum_missing_upstream,kingdom_with_finer_but_missing_phylum
2840134,0,0,0,0,0,0,0


In [32]:
%%sql
SELECT 'genus_snapshot_mismatches' AS check_name, COUNT(*) AS mismatch_count
FROM parents t
JOIN parents g ON g.tax_id = t.t_genus
WHERE t.t_genus IS NOT NULL
  AND (
    t.t_kingdom IS DISTINCT FROM g.t_kingdom
    OR t.t_phylum IS DISTINCT FROM g.t_phylum
    OR t.t_class IS DISTINCT FROM g.t_class
    OR t.t_order IS DISTINCT FROM g.t_order
    OR t.t_family IS DISTINCT FROM g.t_family
  )
UNION ALL
SELECT 'family_snapshot_mismatches', COUNT(*)
FROM parents t
JOIN parents f ON f.tax_id = t.t_family
WHERE t.t_family IS NOT NULL
  AND (
    t.t_kingdom IS DISTINCT FROM f.t_kingdom
    OR t.t_phylum IS DISTINCT FROM f.t_phylum
    OR t.t_class IS DISTINCT FROM f.t_class
    OR t.t_order IS DISTINCT FROM f.t_order
  )
UNION ALL
SELECT 'order_snapshot_mismatches', COUNT(*)
FROM parents t
JOIN parents o ON o.tax_id = t.t_order
WHERE t.t_order IS NOT NULL
  AND (
    t.t_kingdom IS DISTINCT FROM o.t_kingdom
    OR t.t_phylum IS DISTINCT FROM o.t_phylum
    OR t.t_class IS DISTINCT FROM o.t_class
  )
UNION ALL
SELECT 'class_snapshot_mismatches', COUNT(*)
FROM parents t
JOIN parents c ON c.tax_id = t.t_class
WHERE t.t_class IS NOT NULL
  AND (
    t.t_kingdom IS DISTINCT FROM c.t_kingdom
    OR t.t_phylum IS DISTINCT FROM c.t_phylum
  )
UNION ALL
SELECT 'phylum_snapshot_mismatches', COUNT(*)
FROM parents t
JOIN parents ph ON ph.tax_id = t.t_phylum
WHERE t.t_phylum IS NOT NULL
  AND t.t_kingdom IS DISTINCT FROM ph.t_kingdom;

Running query in 'duckdb'

check_name,mismatch_count
genus_snapshot_mismatches,0
family_snapshot_mismatches,0
order_snapshot_mismatches,0
class_snapshot_mismatches,0
phylum_snapshot_mismatches,0


In [33]:
# finest-rank pivot returns 14 rows; default displaylimit is 10.
%config SqlMagic.displaylimit = 20


In [34]:
%%sql
-- finest filled rank + at_rank vs sub_rank (§6)
WITH classified AS (
  SELECT
    tax_id,
    CASE
      WHEN t_species IS NOT NULL THEN 'species'
      WHEN t_genus IS NOT NULL THEN 'genus'
      WHEN t_family IS NOT NULL THEN 'family'
      WHEN t_order IS NOT NULL THEN 'order'
      WHEN t_class IS NOT NULL THEN 'class'
      WHEN t_phylum IS NOT NULL THEN 'phylum'
      WHEN t_kingdom IS NOT NULL THEN 'kingdom'
      ELSE 'none'
    END AS finest_rank,
    CASE
      WHEN t_species IS NOT NULL THEN t_species
      WHEN t_genus IS NOT NULL THEN t_genus
      WHEN t_family IS NOT NULL THEN t_family
      WHEN t_order IS NOT NULL THEN t_order
      WHEN t_class IS NOT NULL THEN t_class
      WHEN t_phylum IS NOT NULL THEN t_phylum
      WHEN t_kingdom IS NOT NULL THEN t_kingdom
      ELSE NULL
    END AS finest_rank_tax_id
  FROM parents
)
SELECT
  finest_rank,
  CASE WHEN tax_id = finest_rank_tax_id THEN 'at_rank' ELSE 'sub_rank' END AS placement,
  COUNT(*) AS n
FROM classified
WHERE finest_rank != 'none'
GROUP BY 1, 2
ORDER BY CASE finest_rank
  WHEN 'kingdom' THEN 1 WHEN 'phylum' THEN 2 WHEN 'class' THEN 3
  WHEN 'order' THEN 4 WHEN 'family' THEN 5 WHEN 'genus' THEN 6
  WHEN 'species' THEN 7 END,
  placement;


Running query in 'duckdb'

finest_rank,placement,n
kingdom,at_rank,45
kingdom,sub_rank,465
phylum,at_rank,116166
phylum,sub_rank,1062
class,at_rank,28532
class,sub_rank,2248
order,at_rank,43650
order,sub_rank,6567
family,at_rank,167488
family,sub_rank,21625


In [35]:
%config SqlMagic.displaylimit = 10


In [36]:
%%sql
SELECT pn.pathway, COUNT(*) AS dangling_nodes
FROM pathway_nodes pn
LEFT JOIN pathway_edges e ON pn.id = e.source OR pn.id = e.target
WHERE e.id IS NULL
GROUP BY pn.pathway
ORDER BY dangling_nodes DESC
LIMIT 10;

Running query in 'duckdb'

pathway,dangling_nodes
1100,3716
1110,2480
1120,1208
1040,135
999,128
1057,127
980,106
998,100
904,99
624,97


## 7. Cross-Domain Joins

Validate joins from RPKM tax_id headers and EC values into the taxonomy and pathway reference tables.

In [37]:
def materialize_rpk_tax_ids(tax_ids: set[str]) -> None:
    ids_int = [int(t) for t in tax_ids]
    # Use a regular session table so the following %%sql cell can see it.
    conn.execute("CREATE OR REPLACE TABLE rpk_tax_ids (tax_id INT)")
    conn.executemany("INSERT INTO rpk_tax_ids VALUES (?)", [(i,) for i in ids_int])


def tax_id_match_rates(tax_ids: set[str], label: str) -> None:
    materialize_rpk_tax_ids(tax_ids)
    r = conn.execute("""
        SELECT
            (SELECT COUNT(*) FROM rpk_tax_ids) AS total_headers,
            (SELECT COUNT(*) FROM rpk_tax_ids r JOIN names n ON r.tax_id = n.tax_id) AS matched_names,
            (SELECT COUNT(DISTINCT r.tax_id) FROM rpk_tax_ids r JOIN names n ON r.tax_id = n.tax_id) AS distinct_tax_ids_with_names,
            (SELECT COUNT(*) FROM rpk_tax_ids r JOIN nodes nd ON r.tax_id = nd.id) AS matched_nodes,
            (SELECT COUNT(*) FROM rpk_tax_ids r JOIN parents p ON r.tax_id = p.tax_id) AS matched_parents
    """).fetchone()
    print(f"--- {label} ---")
    print(f"  tax_id column headers: {r[0]}")
    print(f"  headers with >=1 names row: {r[1]} (distinct tax_ids: {r[2]})")
    print(f"  headers with nodes row: {r[3]}")
    print(f"  headers with parents row: {r[4]}")


tax_id_match_rates(tax_set_1, "test_rpkm_1")
tax_id_match_rates(tax_set_2, "test_rpkm_2")
materialize_rpk_tax_ids(tax_set_1)
print("rpk_tax_ids reset to test_rpkm_1 for the unmapped tax_id sample below.")

--- test_rpkm_1 ---
  tax_id column headers: 8
  headers with >=1 names row: 8 (distinct tax_ids: 8)
  headers with nodes row: 8
  headers with parents row: 8
--- test_rpkm_2 ---
  tax_id column headers: 12
  headers with >=1 names row: 12 (distinct tax_ids: 12)
  headers with nodes row: 12
  headers with parents row: 12
rpk_tax_ids reset to test_rpkm_1 for the unmapped tax_id sample below.


In [38]:
%%sql
SELECT r.tax_id
FROM rpk_tax_ids r
LEFT JOIN names n ON r.tax_id = n.tax_id
WHERE n.tax_id IS NULL
ORDER BY r.tax_id
LIMIT 20;

Running query in 'duckdb'

tax_id


In [39]:
%%sql
WITH normalized AS (
    SELECT DISTINCT
        CASE
            WHEN "EC#" IS NULL OR TRIM(CAST("EC#" AS VARCHAR)) IN ('', 'None', 'none', 'NA', 'null') THEN '0.0.0.0'
            WHEN STARTS_WITH(CAST("EC#" AS VARCHAR), 'EC:') THEN SUBSTR(CAST("EC#" AS VARCHAR), 4)
            ELSE CAST("EC#" AS VARCHAR)
        END AS ec
    FROM rpkm_1
)
SELECT
    (SELECT COUNT(*) FROM normalized) AS distinct_ecs,
    (SELECT COUNT(*) FROM normalized n JOIN pathway_nodes pn ON n.ec = pn.name) AS matched_pathway_nodes;

Running query in 'duckdb'

distinct_ecs,matched_pathway_nodes
9034,3258


## 8. Summary

ER diagram, edge evidence (review flags), regression validation targets, and gotchas: [`analytics/exploration/docs/data-model.md`](../docs/data-model.md).